# 🧪 Лабораторна робота №2
## Тема: Аналогові сигнали на ESP32 — АЦП (ADC) та широтно-імпульсна модуляція (PWM)

## 🎯 Мета роботи:
- Навчитися зчитувати аналогові сигнали (потенціометр, датчики) через `machine.ADC`
- Навчитися формувати ШІМ-сигнал через `machine.PWM`
- Керувати яскравістю світлодіода, звуком зумера та кутом сервопривода
- Поєднати ADC і PWM в одному проєкті (керування виходом за значенням входу)

---

## 📖 Розширені теоретичні відомості

### 1️⃣ Аналоговий сигнал і АЦП
- Цифровий пін розрізняє лише два стани (`0`/`1`), а **аналоговий** сигнал може набувати будь-якого значення в діапазоні напруги (наприклад, від 0 до 3.3 В).
- **АЦП (ADC, Analog-to-Digital Converter)** перетворює напругу на число. ESP32 має 12-бітний АЦП (значення `read()` від 0 до 4095) на пінах `GPIO32–GPIO39` (канали `ADC1`).
- Типове застосування: потенціометр, фоторезистор (LDR), термістор (NTC), аналогові датчики газу/вологості тощо.

### 2️⃣ Клас `machine.ADC`
```python
from machine import ADC, Pin

pot = ADC(Pin(34))
pot.atten(ADC.ATTN_11DB)      # розширює діапазон вимірювання до ~0-3.3В
value = pot.read()            # у старих портах: 0-4095 (12 біт)
value16 = pot.read_u16()      # у новіших портах: 0-65535 (уніфіковано для всіх плат)
```

- `atten()` задає послаблення вхідного сигналу — без нього коректно вимірюється лише до ~1 В.
- Рекомендується використовувати `read_u16()` — він однаково працює на різних портах MicroPython (ESP32, RP2, STM32).

### 3️⃣ Широтно-імпульсна модуляція (PWM)
- PWM — це швидке вмикання/вимикання цифрового піна, де середня «ефективна» напруга залежить від **шпаруватості (duty cycle)** — частки часу, коли сигнал увімкнений.
- Використовується там, де потрібен «аналоговий» ефект на цифровому виході: яскравість LED, гучність/тон зумера, кут повороту сервопривода.
- **Частота (freq)** — скільки разів на секунду повторюється імпульс (для LED зазвичай 1000 Гц, для сервоприводів — 50 Гц).

### 4️⃣ Клас `machine.PWM`
```python
from machine import Pin, PWM

pwm = PWM(Pin(2))
pwm.freq(1000)          # 1000 Гц
pwm.duty(512)           # 0-1023 (стара шкала) — приблизно 50%
pwm.duty_u16(32768)     # 0-65535 (уніфікована шкала) — приблизно 50%
```

- `duty(0)` / `duty_u16(0)` — сигнал повністю вимкнений; `duty(1023)` / `duty_u16(65535)` — сигнал повністю увімкнений (макс. яскравість).
- Щоб зупинити ШІМ і звільнити пін: `pwm.deinit()`.

### 5️⃣ Керування сервоприводом через PWM
- Стандартний сервопривод очікує імпульси з частотою **50 Гц**, де тривалість імпульсу від ~0.5 мс (0°) до ~2.5 мс (180°).
- У шкалі `duty_u16` (0–65535) на частоті 50 Гц (період 20 мс) це приблизно відповідає значенням від ~1600 до ~8100 (залежить від моделі серво — параметри варто підбирати експериментально):
  ```python
  from machine import Pin, PWM

  servo = PWM(Pin(18), freq=50)
  servo.duty_u16(1600)   # приблизно 0°
  servo.duty_u16(4900)   # приблизно 90°
  servo.duty_u16(8100)   # приблизно 180°
  ```

### 6️⃣ Типові помилки початківців
- Забули викликати `atten(ADC.ATTN_11DB)` → значення АЦП «застрягають» біля максимуму навіть при низькій напрузі.
- Плутають шкали `duty()` (0–1023) і `duty_u16()` (0–65535) — використовують значення з іншої шкали.
- Обрали для ADC пін, який не підтримує аналогове читання (не всі GPIO мають ADC).
- Задали для сервопривода занадто малу/велику шпаруватість — механізм «тріщить», намагаючись вийти за фізичні межі повороту.
- Забули `pwm.deinit()` при повторній ініціалізації PWM на тому самому піні → помилка `ValueError: PWM already active`.

### 8️⃣ Як перевіряти код цієї лабораторної роботи

У MicroPython немає локального "пісочного" середовища, як для звичайного Python — код виконується **на самому мікроконтролері** ESP32. Є два способи перевірити свої рішення.

**Варіант A. Jupyter Notebook + реальна плата ESP32 (рекомендовано, якщо є плата)**

1. Прошийте плату прошивкою MicroPython ([https://micropython.org/download/ESP32_GENERIC/](https://micropython.org/download/ESP32_GENERIC/)), один раз через `esptool.py`.
2. Встановіть ядро для Jupyter, яке вміє працювати з платою по USB/Serial:
   ```bash
   pip install jupyter_micropython_kernel
   python -m jupyter_micropython_kernel.install
   ```

3. Відкрийте цей ноутбук і оберіть ядро **MicroPython** (або **MicroPython - USB**) замість Python 3.
4. У першій комірці виконайте підключення до плати (замініть порт на свій — `COMx` у Windows, `/dev/ttyUSB0` або `/dev/ttyACM0` у Linux/macOS):
   ```
   %serialconnect --port=COM5 --baud=115200
   ```

5. Далі виконуйте комірки з кодом як завжди — код одразу запускається на платі, а `print()` повертає результат прямо в ноутбук. Світлодіоди й кнопки реагують фізично.
6. Якщо потрібно скинути з'єднання: `%disconnect`, а щоб побачити файли на платі: `%lsmagic`.

**Варіант B. Немає плати під рукою — симулятор Wokwi**

1. Відкрийте [https://wokwi.com](https://wokwi.com) → **New Project → ESP32 → MicroPython**.
2. У файлі `main.py` вставте код із завдання.
3. На вкладці **diagram.json** опишіть схему (перелік деталей і з'єднань наведено в кожному завданні нижче) — або зберіть її мишкою в графічному редакторі Wokwi, перетягуючи компоненти з панелі **Parts**.
4. Натисніть зелену кнопку **▶ Play** — почнеться симуляція. Вивід `print()` з'являється у вкладці **Serial Monitor**.
5. Перевірте результат за критеріями, вказаними у кожному завданні (що саме має блимати/показуватись/виводитись).
6. Готовий проєкт можна зберегти (**Save**) і надіслати викладачу посилання на нього — це і буде звіт з лабораторної.

**Схема для цієї лабораторної (Wokwi):** потенціометр (Potentiometer), середній вивід — до `GPIO34` (ADC), крайні — до `3V3` і `GND`; світлодіод з резистором 220 Ом на `GPIO2` (для ШІМ-яскравості); п'єзо-зумер (Buzzer) на `GPIO25`; сервопривод (Servo Motor) на `GPIO18` (живлення від `5V`/`GND` симулятора).

*Особливість симуляції ADC у Wokwi:* потенціометр — це віртуальний повзунок, який можна тягнути мишкою прямо під час запущеної симуляції — так само, як обертати реальний.

> 💡 Порада: пишіть і перевіряйте код невеликими фрагментами — спочатку переконайтесь, що плата взагалі відповідає (наприклад, `print("hello")`), і лише потім додавайте логіку.

---

## 📝 Завдання 1. Зчитування потенціометра
Підключіть потенціометр до `GPIO34` і виводьте його «сире» значення (0–65535) у Serial щосекунди.

In [ ]:
from machine import ADC, Pin
import time

pot = ADC(Pin(34))
pot.atten(ADC.ATTN_11DB)

while True:
    print("ADC:", pot.read_u16())
    time.sleep(1)


### 🔢 Завдання:
1. Переведіть значення АЦП у відсотки (0–100%).
2. Переведіть значення АЦП у напругу (0–3.3 В) з округленням до 2 знаків.
3. Виводьте значення лише тоді, коли воно змінилось більш ніж на 5% від попереднього.
4. Знайдіть і виведіть мінімальне й максимальне значення, зафіксовані за час роботи програми.
5. Порахуйте й виведіть середнє значення з останніх 10 вимірювань (згладжування).
6. Додайте світлодіод, що вмикається, коли значення перевищує 50%.
7. Виведіть просту текстову шкалу з символів `#`, довжина якої залежить від значення (наприклад, 20 символів = 100%).
8. Зробіть так, щоб швидкість блимання LED залежала від положення потенціометра.
9. Реалізуйте зчитування з двох потенціометрів (два канали ADC) одночасно.
10. Виведіть, до якої з 4 «зон» (0-25%, 25-50%, 50-75%, 75-100%) відноситься поточне значення.

## 📝 Завдання 2. «Дихання» світлодіода (PWM)
Реалізуйте плавну зміну яскравості світлодіода на `GPIO2` від 0% до 100% і назад («дихання»), використовуючи ШІМ.

In [ ]:
from machine import Pin, PWM
import time

led = PWM(Pin(2))
led.freq(1000)

while True:
    for duty in range(0, 65536, 2048):     # наростання яскравості
        led.duty_u16(duty)
        time.sleep_ms(15)
    for duty in range(65535, -1, -2048):   # згасання яскравості
        led.duty_u16(duty)
        time.sleep_ms(15)


### 🔢 Завдання:
1. Змініть швидкість «дихання» — зробіть його вдвічі повільнішим.
2. Винесіть крок наростання/згасання яскравості в окрему змінну `STEP`.
3. Додайте кнопку, яка призупиняє/відновлює ефект «дихання» (пауза/продовження).
4. Реалізуйте те саме на двох світлодіодах у протифазі (коли один яскравіє, інший тьмяніє).
5. Замініть лінійне наростання на «плавніше» — з використанням квадратичної залежності (`duty**2`).
6. Додайте обмеження мінімальної яскравості — щоб LED ніколи повністю не гасив (мінімум 10%).
7. Зробіть так, щоб кнопка перемикала між трьома швидкостями «дихання».
8. Виведіть у Serial поточну яскравість у відсотках під час ефекту.
9. Реалізуйте функцію `fade(pwm, start, end, step, delay_ms)` і використайте її двічі (вгору/вниз).
10. Додайте зумер, що тихо «пищить» синхронно з піком яскравості (максимальний duty).

## 📝 Завдання 3. Потенціометр керує яскравістю LED
З'єднайте потенціометр (ADC) і світлодіод (PWM): яскравість LED повинна плавно повторювати положення потенціометра.

In [ ]:
from machine import ADC, PWM, Pin
import time

pot = ADC(Pin(34))
pot.atten(ADC.ATTN_11DB)

led = PWM(Pin(2))
led.freq(1000)

while True:
    value = pot.read_u16()
    led.duty_u16(value)
    time.sleep_ms(20)


### 🔢 Завдання:
1. Інвертуйте залежність: чим більше значення потенціометра, тим тьмяніший LED.
2. Додайте «мертву зону» — якщо потенціометр менше 5%, LED повністю вимкнений (не тьмяно світиться).
3. Керуйте яскравістю ДВОХ світлодіодів одним потенціометром, але з різницею у фазі (складіть/відніміть зсув).
4. Додайте другий потенціометр, що керує частотою ШІМ (`freq`), а не яскравістю.
5. Виведіть у Serial одночасно значення потенціометра (%) і яскравість LED (%).
6. Реалізуйте «клацання» по фіксованих рівнях яскравості (0%, 25%, 50%, 75%, 100%) замість плавної зміни.
7. Додайте кнопку, що фіксує (запам'ятовує) поточну яскравість, ігноруючи подальші рухи потенціометра, доки кнопку не відпустять.
8. Додайте згладжування (усереднення останніх 5 зчитувань ADC) для усунення «тремтіння» яскравості.
9. Керуйте кольором RGB-світлодіода (3 канали PWM) трьома потенціометрами.
10. Реалізуйте автоматичний нічний режим: якщо значення ADC менше порогу протягом 3 секунд поспіль — LED вимикається повністю.

## 📝 Завдання 4. Зумер: тон і проста мелодія
Підключіть п'єзо-зумер до `GPIO25` і відтворіть на ньому одну ноту, а потім коротку мелодію з кількох нот, використовуючи `freq()` ШІМ як частоту звуку.

In [ ]:
from machine import Pin, PWM
import time

buzzer = PWM(Pin(25))

def tone(freq_hz, duration_ms):
    buzzer.freq(freq_hz)
    buzzer.duty_u16(30000)     # ~50% гучності
    time.sleep_ms(duration_ms)
    buzzer.duty_u16(0)         # тиша між нотами
    time.sleep_ms(30)

# Ноти (Гц): C4, D4, E4, F4, G4
melody = [262, 294, 330, 349, 392]

for note in melody:
    tone(note, 300)


### 🔢 Завдання:
1. Відтворіть мелодію задом наперед.
2. Додайте паузи (тишу) різної довжини між нотами, використовуючи список кортежів `(нота, тривалість)`.
3. Зробіть так, щоб кнопка запускала відтворення мелодії за одним натисканням.
4. Додайте другу мелодію і перемикання між ними двома кнопками.
5. Реалізуйте «сирену» — плавну зміну частоти від 400 до 1000 Гц і назад у циклі.
6. Зробіть гучність (`duty_u16`) регульованою потенціометром під час відтворення.
7. Відтворіть просту відому мелодію (наприклад «Жили у бабусі» чи «Little Star») повним рядком нот.
8. Додайте функцію `beep(times, freq, duration)` для N однакових коротких сигналів (наприклад, як звук помилки).
9. Виведіть у Serial назву ноти, яка зараз грає.
10. Реалізуйте «піаніно» на 4 кнопках — кожна кнопка відтворює свою фіксовану ноту, доки утримується.

## 📝 Завдання 5. Керування сервоприводом
Підключіть сервопривод до `GPIO18`. Реалізуйте плавний поворот від 0° до 180° і назад, а також визначте функцію `set_angle(degrees)`.

In [ ]:
from machine import Pin, PWM
import time

servo = PWM(Pin(18), freq=50)

MIN_DUTY = 1600     # приблизно 0 градусів (підберіть під свою модель серво)
MAX_DUTY = 8100      # приблизно 180 градусів

def set_angle(degrees):
    degrees = max(0, min(180, degrees))
    duty = int(MIN_DUTY + (MAX_DUTY - MIN_DUTY) * degrees / 180)
    servo.duty_u16(duty)

while True:
    for angle in range(0, 181, 5):
        set_angle(angle)
        time.sleep_ms(20)
    for angle in range(180, -1, -5):
        set_angle(angle)
        time.sleep_ms(20)


### 🔢 Завдання:
1. Керуйте кутом сервопривода напряму потенціометром (0-100% ADC → 0-180°).
2. Додайте дві кнопки: «+10°» і «-10°» для покрокового керування кутом.
3. Виведіть у Serial поточний кут кожного разу, коли він змінюється.
4. Реалізуйте «маятниковий» рух — сервопривод хитається між 60° і 120°.
5. Додайте калібрувальний режим: виводьте значення `duty` для кутів 0°, 90°, 180°, щоб підібрати точні `MIN_DUTY`/`MAX_DUTY`.
6. Керуйте двома сервоприводами одночасно (наприклад, імітація «шиї» камери — панорама і нахил).
7. Реалізуйте плавний рух до заданого кута із заданою швидкістю (градусів за секунду), а не миттєвий стрибок.
8. Додайте кнопку «додому», яка повертає сервопривод у позицію 90° незалежно від поточного положення.
9. Обмежте робочий діапазон сервопривода користувацькими межами (наприклад, 30°–150°) через змінні.
10. Побудуйте «шлагбаум» — сервопривод відкриває прохід (0°→90°) при натисканні кнопки і закриває (90°→0°) через 3 секунди.

---

## Висновок

> В цій ...